# GitHub Founder-Departure Commit Corpus\n\nThis notebook demonstrates the data-standardization script (`data.py`) used to build the **GitHub Founder-Departure Commit Corpus** dataset.\n\nThe full pipeline: 121 real GitHub repositories were sampled via the GitHub REST search API (across JavaScript/Python/Java/Go and 3 popularity strata), each fully cloned locally (`git clone --bare`) and mined with `git log --numstat` for complete per-commit, per-file authorship history. A filter funnel reduced these to 34 'founder-only' candidate repos: >=100 total commits, no history-loss/squash artifact, and a single author holding >=70% share of commits in the first ~50-commit/6-month window.\n\nThis notebook demonstrates the **final standardization step**: turning raw per-(commit,file) rows (as mined by `git log --numstat`) into the `exp_sel_data_out.json` schema, where each example is one (commit, file) row with `input` = observable commit/file-change features (author identity withheld) and `output` = the `founder`/`other` authorship label.\n\n**What determines whether an open-source project survives its founder stepping away?** This dataset is built to let downstream models learn to predict authorship (founder vs. other) from purely observable commit metadata — a proxy for studying bus-factor and founder-dependency risk.

In [ ]:
import subprocess, sys\ndef _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])\n\n# The original data.py uses only the Python stdlib (json, os) -- no third-party\n# packages needed for the core logic. We only need matplotlib for the results\n# visualization cell at the end, which is pre-installed on Colab.\nif 'google.colab' not in sys.modules:\n    _pip('matplotlib==3.10.0')

In [ ]:
# Original imports from data.py, plus matplotlib for the results plot below.\nimport json\nimport os\n\nimport matplotlib.pyplot as plt

## Load the data\n\n`mini_demo_data.json` is a curated subset of 100 raw per-(commit,file) rows (the same row shape that `temp/build_corpus.py` writes to `temp/datasets/github_founder_corpus_rows.jsonl` in the original pipeline), spanning 5 diverse repos. This is the raw input that `data.py`'s `main()` reads line-by-line before standardizing it into the final `exp_sel_data_out.json` schema.\n\nWe try loading from the GitHub raw URL first (works once this repo is pushed / on Colab), and fall back to the local file (works right now).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-24ffbe-pre-departure-bus-factor-diffusion/main/round-1/dataset-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
rows = load_data()
print(f"loaded {len(rows)} raw (commit,file) rows")
print(json.dumps(rows[0], indent=2))

## Config\n\nThe only tunable parameter in the original `data.py` is `PER_REPO_CAP`: repos with more than this many (commit,file) rows get systematically strided down (every Nth row, chronological order preserved) so a few huge-history repos (e.g. `jenkinsci/jenkins`) can't dominate the corpus.\n\nIn the full pipeline this is `4000`. Our `mini_demo_data.json` only has 20 rows per repo, so we set the cap much lower here (`5`) purely so the striding logic actually gets exercised on this tiny sample -- with the original `4000` cap, no row in our 20-row-per-repo sample would ever be dropped. Bump `PER_REPO_CAP` back up to `4000` to reproduce the original pipeline's behavior on the full dataset.

In [ ]:
# PER_REPO_CAP = 4000  # original pipeline value (stratified cap, see data.py)
PER_REPO_CAP = 5  # demo value: small enough that the stride logic actually fires on our 20-rows-per-repo mini sample

## `to_example`: standardize one raw row into the output schema\n\nThis is copied as-is from `data.py`. It builds the `input` feature JSON (author identity withheld) and the `founder`/`other` `output` label, plus the `metadata_*` provenance fields, from one raw (commit,file) row.

In [ ]:
def to_example(row):
    # `input`: the observable commit/file-change features a downstream DOA /
    # truck-factor / survival model would condition on. Author identity itself
    # is withheld from `input` since `output` is the founder/non-founder label
    # derived from it -- author identity is still preserved as metadata for
    # provenance and alias-resolution auditing.
    input_obj = {
        "commit_index": row["commit_index"],
        "n_commits_total": row["n_commits_total"],
        "days_since_repo_created": row["days_since_repo_created"],
        "file_path": row["file_path"],
        "file_ext": row["file_ext"],
        "lines_added": row["lines_added"],
        "lines_removed": row["lines_removed"],
        "is_creation": row["is_creation"],
        "repo_stars": row["stars"],
        "repo_forks": row["forks"],
        "repo_primary_language": row["primary_language"],
    }
    output = "founder" if row["is_founder_commit"] == 1 else "other"
    example = {
        "input": json.dumps(input_obj, ensure_ascii=False),
        "output": output,
        "metadata_repo_id": row["repo_id"],
        "metadata_full_name": row["full_name"],
        "metadata_license": row["license"],
        "metadata_repo_created_at": row["repo_created_at"],
        "metadata_commit_sha": row["commit_sha"],
        "metadata_commit_timestamp": row["commit_timestamp"],
        "metadata_author_alias_key": row["author_alias_key"],
        "metadata_author_email": row["author_email"],
        "metadata_author_name": row["author_name"],
        "metadata_dominant_founder_share_first_window": row["dominant_founder_share_first_window"],
        "metadata_alias_ambiguous_repo": row["alias_ambiguous_repo"],
        "metadata_task_type": "classification",
        "metadata_n_classes": 2,
    }
    return example

## Standardize the corpus\n\nThis mirrors `main()` in `data.py`, adapted to iterate over the in-memory `rows` list loaded above instead of reading `temp/datasets/github_founder_corpus_rows.jsonl` line-by-line. Logic is unchanged:\n\n1. **First pass**: count rows per repo (`full_name`) to compute a per-repo stride.\n2. **Second pass**: keep every Nth row per repo (`i % strides[name] != 0` skips a row), preserving chronological order, and convert each kept row via `to_example`.\n3. Wrap the examples in the `exp_sel_data_out.json` metadata/datasets envelope.

In [ ]:
# First pass: count rows per repo so the systematic-stride sampling below
# can pick every Nth row per repo (preserving chronological spread and
# founder/non-founder mix) rather than truncating to the earliest rows.
counts = {}
for row in rows:
    full_name = row["full_name"]
    counts[full_name] = counts.get(full_name, 0) + 1

strides = {name: max(1, n // PER_REPO_CAP + 1) for name, n in counts.items()}

examples = []
seen = {}
for row in rows:
    name = row["full_name"]
    i = seen.get(name, 0)
    seen[name] = i + 1
    if i % strides[name] != 0:
        continue
    examples.append(to_example(row))

out = {
    "metadata": {
        "source": "Local git clone (git log --numstat) over GitHub repos sampled via "
                   "the GitHub REST search/repositories API across JavaScript/Python/Java/Go "
                   "and 3 popularity strata (100-1k, 1k-10k, 10k+ stars); repo-level metadata "
                   "(stars, forks, license, language, created_at) from the same API.",
        "description": "Per-(commit,file) rows for GitHub repos passing founder-only-start "
                        "filters (>=100 commits, no history-loss/squash artifact, a single "
                        "author holding >=70% share of commits in the first ~50-commit / "
                        "6-month window). `output` is founder-vs-other authorship of that "
                        "commit; `input` withholds author identity so it can serve as a "
                        "downstream classification/DOA feature set without leaking the label. "
                        f"Repos with more than {PER_REPO_CAP} (commit,file) rows are systematically "
                        "strided down to that cap (keep every Nth row, chronological order preserved) "
                        "to keep the corpus size bounded and prevent a few huge-history repos "
                        "(e.g. jenkinsci/jenkins) from dominating the example count.",
        "n_examples": len(examples),
        "n_repos": len({e["metadata_full_name"] for e in examples}),
    },
    "datasets": [
        {
            "dataset": "github_founder_departure_corpus",
            "examples": examples,
        }
    ],
}

print(f"standardized {len(examples)} examples across "
      f"{out['metadata']['n_repos']} repos (from {len(rows)} raw rows)")

## Results\n\nA quick look at the standardized output: per-repo example counts, the founder/other label balance, and one full example.

In [ ]:
from collections import Counter

per_repo_counts = Counter(e["metadata_full_name"] for e in examples)
label_counts = Counter(e["output"] for e in examples)

print(f"{'repo':35s} {'examples':>8s}")
for name, n in per_repo_counts.most_common():
    print(f"{name:35s} {n:8d}")

print()
print("label balance:", dict(label_counts))
print()
print("one standardized example:")
print(json.dumps(examples[0], indent=2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

repo_names = list(per_repo_counts.keys())
axes[0].bar(range(len(repo_names)), [per_repo_counts[n] for n in repo_names], color="#4C72B0")
axes[0].set_xticks(range(len(repo_names)))
axes[0].set_xticklabels([n.split("/")[-1] for n in repo_names], rotation=45, ha="right")
axes[0].set_ylabel("standardized examples")
axes[0].set_title("Examples per repo (after stride-cap sampling)")

labels = list(label_counts.keys())
axes[1].bar(labels, [label_counts[l] for l in labels], color=["#55A868", "#C44E52"])
axes[1].set_ylabel("count")
axes[1].set_title("founder vs. other authorship label balance")

plt.tight_layout()
plt.show()